# Facepoints dataset exploration

This notebook inspects the face-landmark dataset, visualises the annotated keypoints, reproduces the seeded training split and calculates channel-wise `MEAN` and `STD`. The statistics are calculated at the same **100 × 100** resolution used by the model and are saved to `outputs/normalization.json` for training and inference.


In [ ]:
from pathlib import Path
import sys
import json
import shutil
import tempfile

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from torchvision import transforms
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "public_tests" / "00_test_img_input" / "train"
IMAGES_DIR = DATA_DIR / "images"
GT_PATH = DATA_DIR / "gt.csv"
NORMALIZATION_PATH = PROJECT_ROOT / "outputs" / "normalization.json"
IMAGE_SIZE = (100, 100)
BATCH_SIZE = 64
TRAIN_FRACTION = 0.8
SEED = 42
N_IMAGES_TO_DISPLAY = 4

from facepoints.data.dataset import read_csv
from facepoints.utils import save_normalization

if not IMAGES_DIR.is_dir() or not GT_PATH.is_file():
    raise FileNotFoundError(
        f"Dataset paths are not valid. Set IMAGES_DIR and GT_PATH in this cell.\nImages: {IMAGES_DIR}\nAnnotations: {GT_PATH}"
    )

train_gt = read_csv(GT_PATH)
sample_names = sorted(train_gt)
print(f"Annotated images: {len(sample_names):,}")
print(f"Images directory: {IMAGES_DIR}")
print(f"Ground-truth file: {GT_PATH}")


## Dataset dimensions

The next cell reports the source image dimensions. The model preprocessing resizes every image to the configured `IMAGE_SIZE`; original dimensions are retained for coordinate restoration during inference.


In [ ]:
image_dimensions = {}
for name in sample_names:
    with Image.open(IMAGES_DIR / name) as image:
        image_dimensions[name] = image.size

unique_dimensions, dimension_counts = np.unique(
    np.asarray(list(image_dimensions.values())), axis=0, return_counts=True
)
for dimension, count in zip(unique_dimensions, dimension_counts):
    print(f"{int(dimension[0])} × {int(dimension[1])}: {count} images")


## Image and ground-truth visualisation

Landmark indices are displayed without connecting lines because the source annotations do not define a facial-contour topology. Coordinates are shown in the original image pixel space.


In [ ]:
from facepoints.visualization import visualize_keypoints

for name in sample_names[:N_IMAGES_TO_DISPLAY]:
    with Image.open(IMAGES_DIR / name) as source:
        image = source.convert("RGB")
    figure, _ = visualize_keypoints(
        image,
        keypoints=train_gt[name],
        title=name,
        show_indices=True,
    )
    plt.show()
    plt.close(figure)


## Reproducible train/validation split

The split matches the original training code: a permutation generated with `SEED` is created, and the first `round(N × 0.8)` indices form the training subset. Only these training images are used to calculate the channel statistics.


In [ ]:
train_indices = np.random.RandomState(SEED).permutation(len(sample_names))
train_indices = train_indices[:round(len(sample_names) * TRAIN_FRACTION)]
validation_count = len(sample_names) - len(train_indices)
print(f"Training images: {len(train_indices):,}")
print(f"Validation images: {validation_count:,}")


## Calculate MEAN and STD

The calculation follows the supplied implementation: RGB conversion, bicubic resize, `ToTensor()`, accumulation of channel sums and squared sums in `float64`, and population standard deviation. The image resolution is `100 × 100`, as agreed. This cell intentionally requires CUDA, matching the original statistics-calculation procedure.


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. Run this statistics cell in a CUDA-enabled environment.")

device = torch.device("cuda")
transform = transforms.ToTensor()
channel_sum = torch.zeros(3, dtype=torch.float64, device=device)
channel_squared_sum = torch.zeros(3, dtype=torch.float64, device=device)
num_pixels = 0

with tempfile.TemporaryDirectory(prefix="facepoints_") as local_images_path:
    shutil.copytree(IMAGES_DIR, local_images_path, dirs_exist_ok=True)
    for start in tqdm(range(0, len(train_indices), BATCH_SIZE), desc="Calculating MEAN and STD"):
        images = []
        for index in train_indices[start:start + BATCH_SIZE]:
            image_path = Path(local_images_path) / sample_names[index]
            with Image.open(image_path) as sample:
                sample = sample.convert("RGB")
                sample = sample.resize(
                    (IMAGE_SIZE[1], IMAGE_SIZE[0]),
                    resample=Image.Resampling.BICUBIC,
                )
                images.append(transform(sample))
        batch = torch.stack(images).to(device)
        channel_sum += batch.sum(dim=(0, 2, 3), dtype=torch.float64)
        channel_squared_sum += batch.square().sum(dim=(0, 2, 3), dtype=torch.float64)
        num_pixels += batch.shape[0] * IMAGE_SIZE[0] * IMAGE_SIZE[1]

mean = channel_sum / num_pixels
std = torch.sqrt(
    torch.clamp(
        channel_squared_sum / num_pixels - mean.square(),
        min=torch.finfo(torch.float32).eps,
    )
)
MEAN = tuple(mean.cpu().tolist())
STD = tuple(std.cpu().tolist())

print("MEAN =", MEAN)
print("STD =", STD)


## Save normalisation metadata

`train.py` and `notebooks/inference.ipynb` load this artifact directly. Re-run the statistics cell whenever the dataset, training split, seed or input resolution changes.


In [ ]:
normalization = save_normalization(
    NORMALIZATION_PATH,
    mean=MEAN,
    std=STD,
    image_size=IMAGE_SIZE,
    num_images=len(train_indices),
    seed=SEED,
    train_fraction=TRAIN_FRACTION,
)
print(f"Saved normalisation metadata to: {NORMALIZATION_PATH}")
print(json.dumps(normalization, indent=4))
